# K-Means + PCA — Wine Quality (red)

Notebook de ejemplo: **K-Means + PCA** con datos reales. Las etiquetas del CSV (si existen) solo sirven para **validación externa**.

## Pasos principales

| Paso | Sección | Qué haces |
|------|---------|-----------|
| **0** | Helpers | Preprocesado, `Pipeline`, métricas internas (silhouette) y externas (ARI). |
| **1** | Explorar CSV | Columnas, tipos, faltantes. |
| **2** | CONFIG | Rutas, `LABEL_COL` (opcional), `N_CLUSTERS`, PCA. |
| **3** | Carga | Leer CSV. |
| **4** | EDA | Distribuciones y correlaciones. |
| **5** | Features | Matriz **X** (sin usar `LABEL_COL` en el fit). |
| **6** | Preprocesado | `ColumnTransformer` (imputer + escalar / one-hot). |
| **7** | PCA | `Pipeline(preprocess → PCA)`; varianza explicada y vista 2D. |
| **8** | K-Means | Método del codo, silhouette; `Pipeline(preprocess → KMeans)`. |
| **9** | Resultados | Asignación de clusters, gráficos y métricas opcionales vs `LABEL_COL`. |

**Pipeline:** el preprocesado se ajusta **dentro** de cada estimador (`fit` del `Pipeline`), igual que en los ejemplos supervisados de [07.b](../07.b-ejemplos-supervisados/).

> Ejecuta Jupyter desde `07.c-ejemplos-no-supervisados/`.

**Plantilla:** [01-clustering-kmeans-pca.ipynb](../01-clustering-kmeans-pca.ipynb)


In [ ]:
# =============================================================================
# Helpers — preprocesado y clustering (sin target en el fit)
# =============================================================================
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")


def infer_feature_columns(df, drop_cols, feature_cols, label_col=None):
    """Columnas para clustering (X). Excluye LABEL_COL y DROP_COLS."""
    if feature_cols is not None:
        return list(feature_cols)
    exclude = set(drop_cols)
    if label_col:
        exclude.add(label_col)
    return [c for c in df.columns if c not in exclude]


def infer_column_types(X, numeric_cols=None, categorical_cols=None):
    if numeric_cols is None:
        numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    if categorical_cols is None:
        categorical_cols = X.select_dtypes(
            include=["object", "category", "bool", "string"]
        ).columns.tolist()
    return list(numeric_cols), list(categorical_cols)


def build_preprocess(numeric_cols, categorical_cols):
    """ColumnTransformer: imputer + escalar / one-hot (mismo patrón que 07.b)."""
    transformers = []
    if numeric_cols:
        transformers.append(
            (
                "num",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_cols,
            )
        )
    if categorical_cols:
        transformers.append(
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "encoder",
                            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                        ),
                    ]
                ),
                categorical_cols,
            )
        )
    if not transformers:
        raise ValueError("No hay columnas numéricas ni categóricas para preprocesar.")
    return ColumnTransformer(transformers, remainder="drop")


def read_csv_checked(path, sep, required_cols=None):
    df = pd.read_csv(path, sep=sep)
    if df.shape[1] == 1:
        raise ValueError(
            f"Solo 1 columna con sep={sep!r}. CSV_SEP debe coincidir con PREVIEW_SEP."
        )
    if required_cols:
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise ValueError(f"Faltan columnas {missing}. Columnas: {list(df.columns)}")
    return df


def choose_k_elbow(models_preprocess, X, k_range, random_state):
    """Inercia de KMeans dentro de Pipeline(preprocess → KMeans) por cada K."""
    inertias = []
    for k in k_range:
        pipe = Pipeline(
            [
                ("preprocess", models_preprocess),
                (
                    "kmeans",
                    KMeans(n_clusters=k, random_state=random_state, n_init=10),
                ),
            ]
        )
        pipe.fit(X)
        inertias.append(pipe.named_steps["kmeans"].inertia_)
    return list(k_range), inertias


def internal_cluster_metrics(X_transformed, labels):
    """Métricas sin etiquetas verdaderas (espacio ya preprocesado)."""
    n_labels = len(set(labels))
    if n_labels < 2 or n_labels >= len(labels):
        return {}
    return {
        "silhouette": silhouette_score(X_transformed, labels),
        "calinski_harabasz": calinski_harabasz_score(X_transformed, labels),
        "davies_bouldin": davies_bouldin_score(X_transformed, labels),
    }


def external_cluster_metrics(y_true, labels):
    """Solo si tienes LABEL_COL de referencia (no usada en el fit)."""
    return {"adjusted_rand_index": adjusted_rand_score(y_true, labels)}



## 1. Explorar el CSV (antes de CONFIG)

Ajusta `PREVIEW_PATH` y `PREVIEW_SEP`; revisa tipos y una posible `LABEL_COL`.


In [ ]:
# --- Paso 1: exploración rápida (mismo path/separador que CONFIG) ---
PREVIEW_PATH = "../data/wine_quality_red.csv"
PREVIEW_SEP = ";"

df_preview = pd.read_csv(PREVIEW_PATH, sep=PREVIEW_SEP)
print("Filas:", len(df_preview), "| Columnas:", df_preview.shape[1])
display(df_preview.head())
print("\n--- Tipos ---")
print(df_preview.dtypes)
print("\n--- Faltantes ---")
miss = df_preview.isna().sum()
miss = miss[miss > 0]
if len(miss):
    display(miss.to_frame("n_missing"))
else:
    print("No hay valores faltantes.")

_num = df_preview.select_dtypes(include=[np.number]).columns.tolist()
_cat = df_preview.select_dtypes(include=["object", "category", "bool", "string"]).columns.tolist()
print("\n--- Sugerencia de tipos ---")
print("Numéricas:", _num)
print("Categóricas:", _cat)

if "quality" in df_preview.columns:
    print("\n--- Posible LABEL_COL (solo validación) ---")
    print(df_preview["quality"].value_counts().head(10))

print("\n>>> Siguiente: paso 2 CONFIG.")


## 2. CONFIG — adaptar a tu dataset


In [ ]:
# ========== Paso 2: CONFIG ==========
DATA_PATH = '../data/wine_quality_red.csv'
CSV_SEP = ';'

# Solo para comparar clusters con la realidad (NO entra en X ni en el Pipeline de fit)
LABEL_COL = 'quality'

DROP_COLS = []
FEATURE_COLS = None
NUMERIC_COLS = None
CATEGORICAL_COLS = None

N_CLUSTERS = None  # None → elegir K con el codo (paso 8)
K_RANGE = range(2, 12)
N_COMPONENTS_PCA = 2
RANDOM_STATE = 42


## 3. Carga de datos


In [ ]:
# --- Paso 3: carga ---
df = read_csv_checked(DATA_PATH, CSV_SEP)
print(f"{len(df):,} filas × {df.shape[1]} columnas")
df.head()


## 4. EDA


In [ ]:
# --- Paso 4: EDA breve ---
display(df.describe(include="all").T.head(20))
if LABEL_COL and LABEL_COL in df.columns:
    fig, ax = plt.subplots(figsize=(7, 4))
    vc = df[LABEL_COL].value_counts()
    vc.plot(kind="bar", ax=ax, color="steelblue", edgecolor="k")
    ax.set_title(f"Distribución de {LABEL_COL} (referencia, no usada en fit)")
    ax.set_ylabel("conteo")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


## 5. Matriz de features (X)

Solo columnas predictoras. `LABEL_COL` queda fuera del clustering.


In [ ]:
# --- Paso 5: X y etiquetas opcionales de referencia ---
feature_cols = infer_feature_columns(
    df, DROP_COLS, FEATURE_COLS, label_col=LABEL_COL
)
X = df[feature_cols]
numeric_cols, categorical_cols = infer_column_types(
    X, NUMERIC_COLS, CATEGORICAL_COLS
)
print("Features:", len(feature_cols), "| Num:", len(numeric_cols), "| Cat:", len(categorical_cols))

y_ref = None
if LABEL_COL and LABEL_COL in df.columns:
    y_ref = df[LABEL_COL]
    if y_ref.dtype == "object" or str(y_ref.dtype) == "category":
        y_ref = pd.Categorical(y_ref).codes
    print(f"LABEL_COL={LABEL_COL!r} reservada para validación externa (n={len(y_ref)})")


## 6. Preprocesado (`ColumnTransformer`)

Mismo objeto que alimentará los `Pipeline` de PCA y K-Means.


In [ ]:
# --- Paso 6: preprocesador ---
preprocess = build_preprocess(numeric_cols, categorical_cols)
preprocess


## 7. PCA (`Pipeline`: preprocesado → PCA)

Reduce dimensionalidad **después** de escalar/codificar. Útil para visualizar y para K-Means en espacio compacto.


In [ ]:
# --- Paso 7: PCA dentro de Pipeline ---
pipe_pca = Pipeline(
    [
        ("preprocess", preprocess),
        ("pca", PCA(n_components=N_COMPONENTS_PCA, random_state=RANDOM_STATE)),
    ]
)
X_pca = pipe_pca.fit_transform(X)
pca_step = pipe_pca.named_steps["pca"]

print("Varianza explicada por componente:", np.round(pca_step.explained_variance_ratio_, 4))
print("Varianza acumulada:", round(pca_step.explained_variance_ratio_.sum(), 4))
print("Shape X_pca:", X_pca.shape)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(range(1, len(pca_step.explained_variance_ratio_) + 1), pca_step.explained_variance_ratio_)
axes[0].set_xlabel("componente")
axes[0].set_ylabel("varianza explicada")
axes[0].set_title("Scree (PCA)")

if X_pca.shape[1] >= 2:
    sc = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y_ref if y_ref is not None else "gray", cmap="tab10", alpha=0.7, s=25)
    axes[1].set_xlabel("PC1")
    axes[1].set_ylabel("PC2")
    title = "PC1 vs PC2"
    if y_ref is not None:
        title += f" (color = {LABEL_COL})"
    axes[1].set_title(title)
    if y_ref is not None:
        plt.colorbar(sc, ax=axes[1], label=LABEL_COL)
plt.tight_layout()
plt.show()


## 8. K-Means (`Pipeline`: preprocesado → KMeans)

Método del **codo** y **silhouette** para elegir K si `N_CLUSTERS` es `None`.


In [ ]:
# --- Paso 8: elegir K y ajustar K-Means ---
ks, inertias = choose_k_elbow(preprocess, X, K_RANGE, RANDOM_STATE)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ks, inertias, "o-", color="steelblue")
ax.set_xlabel("K")
ax.set_ylabel("inercia (within-cluster SS)")
ax.set_title("Método del codo — K-Means en Pipeline")
plt.tight_layout()
plt.show()

# Silhouette por K (en espacio preprocesado, sin PCA)
sil_rows = []
for k in ks:
    pipe_k = Pipeline(
        [
            ("preprocess", preprocess),
            ("kmeans", KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)),
        ]
    )
    labels_k = pipe_k.fit_predict(X)
    X_t = pipe_k.named_steps["preprocess"].transform(X)
    sil_rows.append({"K": k, **internal_cluster_metrics(X_t, labels_k)})
sil_df = pd.DataFrame(sil_rows)
display(sil_df.round(4))

if N_CLUSTERS is None:
    best_k = int(sil_df.loc[sil_df["silhouette"].idxmax(), "K"])
    print(f"N_CLUSTERS=None → se elige K={best_k} (máx. silhouette)")
else:
    best_k = int(N_CLUSTERS)
    print(f"N_CLUSTERS fijado en CONFIG: K={best_k}")

pipe_kmeans = Pipeline(
    [
        ("preprocess", preprocess),
        ("kmeans", KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)),
    ]
)
cluster_labels = pipe_kmeans.fit_predict(X)
X_scaled = pipe_kmeans.named_steps["preprocess"].transform(X)
print("\nMétricas internas (K final):")
display(pd.DataFrame([internal_cluster_metrics(X_scaled, cluster_labels)]).round(4))
print(f"Inercia: {pipe_kmeans.named_steps['kmeans'].inertia_:.2f}")


## 9. Resultados y validación opcional

Visualización de clusters. Si definiste `LABEL_COL`, **ARI** compara con la verdad (sin haberla usado en el fit).


In [ ]:
# --- Paso 9: resultados ---
df_out = df.copy()
df_out["cluster"] = cluster_labels

print("Tamaño por cluster:")
display(df_out["cluster"].value_counts().sort_index().to_frame("n"))

if y_ref is not None:
    ext = external_cluster_metrics(y_ref, cluster_labels)
    print(f"\nValidación externa vs {LABEL_COL}:")
    display(pd.DataFrame([ext]).round(4))

# Vista 2D: PCA del mismo Pipeline ajustado en paso 7, coloreado por cluster
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap="tab10", alpha=0.75, s=30, edgecolors="k", linewidths=0.2
)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title(f"K-Means (K={best_k}) en espacio PCA")
plt.colorbar(scatter, ax=ax, label="cluster")
plt.tight_layout()
plt.show()

# Heatmap: crosstab cluster vs etiqueta real
if y_ref is not None and LABEL_COL in df.columns:
    ct = pd.crosstab(df_out["cluster"], df[LABEL_COL], normalize="index")
    fig, ax = plt.subplots(figsize=(8, max(3, len(ct) * 0.5)))
    sns.heatmap(ct, annot=True, fmt=".2f", cmap="Blues", ax=ax)
    ax.set_title(f"Proporción de {LABEL_COL} dentro de cada cluster")
    plt.tight_layout()
    plt.show()
